# XGBoost Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Archivos necesarios en la misma carpeta que este notebook:**
- `xgboost_cloud_best.ubj` — modelo entrenado
- `xgboost_cloud_metadata.json` — metadata (feature_cols, best_cv_auc, etc.)

**Flujo:**
1. Carga el modelo guardado desde Kaggle
2. Lee `features_test.parquet` localmente
3. Genera predicciones
4. Guarda `submission_cloud_xgboost.csv`
5. Sube a Kaggle y recupera AUC público

In [1]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import xgboost as xgb
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.metrics import roc_auc_score

print(f'XGBoost version: {xgb.__version__}')
print('Imports OK')

XGBoost version: 2.1.3
Imports OK


In [2]:
# ─── Paths ────────────────────────────────────────────────────────────────────
# Archivos del modelo: en la misma carpeta que este notebook
MODEL_DIR = Path('.')

# Datos de test: ruta local relativa al proyecto
DATA_DIR  = Path('../../data/processed')

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
print(f'DATA_DIR  : {DATA_DIR.resolve()}')

# Verificar que los archivos del modelo existen
model_file = MODEL_DIR / 'xgboost_cloud_best.ubj'
meta_file  = MODEL_DIR / 'xgboost_cloud_metadata.json'

for f in [model_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle Output tab'
    print(f'  {f.name} : {status}')

MODEL_DIR : C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\cloud\xgboost
DATA_DIR  : C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\data\processed
  xgboost_cloud_best.ubj : ✓ encontrado
  xgboost_cloud_metadata.json : ✓ encontrado


In [3]:
# ─── Cargar metadata ──────────────────────────────────────────────────────────
with open(meta_file, 'r') as f:
    meta = json.load(f)

print('Metadata del modelo:')
print(f'  Train AUC       : {meta["train_auc"]}')
print(f'  CV AUC (Optuna) : {meta["best_cv_auc"]}')
print(f'  n_rounds        : {meta["best_n_rounds"]}')
print(f'  n_features      : {len(meta["feature_cols"])}')
print(f'  GPU usado       : {meta["use_gpu"]}')
print(f'  XGBoost version : {meta["xgboost_version"]}')
print(f'  Timestamp       : {meta["timestamp"]}')
print('  Hiperparámetros:')
for k, v in meta['best_params'].items():
    print(f'    {k:<22}: {v}')

Metadata del modelo:
  Train AUC       : 0.7854
  CV AUC (Optuna) : 0.7682351442895563
  n_rounds        : 998
  n_features      : 30
  GPU usado       : True
  XGBoost version : 3.1.3
  Timestamp       : 2026-02-25T03:01:44.996031
  Hiperparámetros:
    max_depth             : 2.0
    learning_rate         : 0.05388108577817234
    subsample             : 0.5171942605576092
    colsample_bytree      : 0.954660201039391
    min_child_weight      : 3.0
    gamma                 : 3.31261142176991
    reg_alpha             : 6.388511557344611e-06
    reg_lambda            : 0.0004793052550782129


In [4]:
# ─── Cargar modelo ────────────────────────────────────────────────────────────
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
model = xgb.Booster()
model.load_model(model_file)
print('Modelo cargado ✓')

Cargando modelo: xgboost_cloud_best.ubj  (0.90 MB)...
Modelo cargado ✓


In [5]:
# ─── Cargar datos de test ─────────────────────────────────────────────────────
print('Cargando features_test.parquet...')
df_test     = pd.read_parquet(DATA_DIR / 'features_test.parquet')
sk_ids_test = df_test['SK_ID_CURR'].values
feature_cols = meta['feature_cols']

X_test = df_test[feature_cols].values
dtest  = xgb.DMatrix(X_test, feature_names=feature_cols)

print(f'  Test shape  : {X_test.shape}')
print(f'  Features    : {len(feature_cols)}')
print(f'  NaNs en X   : {np.isnan(X_test).sum():,}')

Cargando features_test.parquet...
  Test shape  : (48744, 30)
  Features    : 30
  NaNs en X   : 196,589


In [6]:
# ─── Predicciones ─────────────────────────────────────────────────────────────
print('Generando predicciones...')
y_test_prob = model.predict(dtest)

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

Generando predicciones...
  Predicciones generadas : 48,744
  Score mínimo           : 0.00961
  Score máximo           : 0.98416
  Score medio            : 0.39211
  % predicho como default: 30.69%


In [7]:
# ─── Guardar submission ───────────────────────────────────────────────────────
submission  = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path    = DATA_DIR / 'submission_cloud_xgboost.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

Submission guardado: ..\..\data\processed\submission_cloud_xgboost.csv
Shape: (48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.283752
1,100005,0.658237
2,100013,0.196238
3,100028,0.295674
4,100038,0.637603


## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).  
Usa la Python API (`KaggleApi`) directamente — polling cada 10 s, máx 5 min.

> **Límite**: 5 submissions/día.

In [8]:
# ─── Kaggle submission ────────────────────────────────────────────────────────
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'
_msg = (f"XGBoost Cloud | "
        f"CV AUC={meta['best_cv_auc']:.5f} | "
        f"train AUC={meta['train_auc']:.5f} | "
        f"rounds={meta['best_n_rounds']}")

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    """Poll hasta que aparezca el public_score en la submission correspondiente."""
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(
    file_name   = str(out_path),
    message     = _msg,
    competition = COMPETITION
)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 55)
print(f'RESULTADO KAGGLE')
print(f'=' * 55)
print(f'  AUC test Public  LB  : {public_auc}')
print(f'  AUC test Private LB  : {private_auc}')
print(f'  CV AUC (entrenamiento): {meta["best_cv_auc"]:.5f}')
if public_auc:
    print(f'  Gap CV - Public LB   : {abs(meta["best_cv_auc"] - public_auc):.5f}')
print(f'=' * 55)

Enviando: XGBoost Cloud | CV AUC=0.76824 | train AUC=0.78540 | rounds=998


100%|██████████| 888k/888k [00:01<00:00, 586kB/s]  


Esperando scoring (poll 10 s / máx 5 min)...
  [  0s] status='SubmissionStatus.PENDING'  public_score=''
  [ 10s] status='SubmissionStatus.COMPLETE'  public_score='0.78103'

RESULTADO KAGGLE
  AUC test Public  LB  : 0.78103
  AUC test Private LB  : 0.76993
  CV AUC (entrenamiento): 0.76824
  Gap CV - Public LB   : 0.01279
